# Comparing SC weight parameterisations: Dirichlet vs softmax-Normal

CausalPy offers two model classes for Bayesian {term}`synthetic control` that both produce simplex-valued weights (positive, sum to one) but encode different prior beliefs:

- **Dirichlet** ({class}`~causalpy.pymc_models.WeightedSumFitter`): the concentration parameter `a` controls sparsity vs uniformity. With `a=1` the prior is uniform on the simplex.
- **Softmax-Normal** ({class}`~causalpy.pymc_models.SoftmaxWeightedSumFitter`): places Normal priors on unconstrained logits and maps them to the simplex via softmax. The prior scale `sigma` continuously interpolates between DiD-like uniform weights (small sigma) and SC-like sparse weights (large sigma).

The softmax-Normal parameterisation is the Bayesian analogue of the $\ell_2$ regularisation parameter $\zeta$ in the frequentist Synthetic Difference-in-Differences of {cite:t}`arkhangelsky2021synthetic`.

This notebook is a focused companion to {doc}`sc_pymc_brexit` — see that notebook for the full {term}`synthetic control` pipeline including donor pool selection, diagnostics, and the convex hull assumption.

In [ ]:
import arviz as az
import causalpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pymc_extras.prior import Prior

%config InlineBackend.figure_format = "retina"
seed = 123

## Setup

We use the Brexit GDP dataset with 15 donor countries. For details on donor pool selection, see {doc}`sc_pymc_brexit`.

In [ ]:
df = (
    cp.load_data("brexit")
    .assign(Time=lambda x: pd.to_datetime(x["Time"]))
    .set_index("Time")
    .loc[lambda x: x.index >= "2009-01-01"]
    .drop(["Japan", "Italy", "US", "Spain", "Portugal"], axis=1)
)

treatment_time = pd.to_datetime("2016 June 24")
target_country = "UK"
other_countries = [c for c in df.columns if c != target_country]

sample_kwargs = {
    "tune": 1000,
    "draws": 1000,
    "chains": 2,
    "random_seed": seed,
    "target_accept": 0.95,
}

print(f"Donors ({len(other_countries)}): {other_countries}")
df.head()

## Head-to-head: Dirichlet vs softmax-Normal

We fit both models on the same data with identical `sample_kwargs`. The only difference is the weight prior.

In [ ]:
result_dirichlet = cp.SyntheticControl(
    df,
    treatment_time,
    control_units=other_countries,
    treated_units=[target_country],
    model=cp.pymc_models.WeightedSumFitter(sample_kwargs=sample_kwargs),
)

In [ ]:
result_softmax = cp.SyntheticControl(
    df,
    treatment_time,
    control_units=other_countries,
    treated_units=[target_country],
    model=cp.pymc_models.SoftmaxWeightedSumFitter(sample_kwargs=sample_kwargs),
)

### Posterior weights

Side-by-side posterior median weights with 94% credible intervals. Both models produce simplex weights, but the shape of the posterior distribution over those weights can differ substantially.

In [ ]:
def _extract_weights(result, treated_unit):
    """Extract posterior weight statistics from a SyntheticControl result."""
    beta = result.idata.posterior["beta"].sel(treated_units=treated_unit)
    median = beta.median(dim=["chain", "draw"]).values
    hdi = az.hdi(beta, hdi_prob=0.94)["beta"].values
    return median, hdi[:, 0], hdi[:, 1]


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
donors = other_countries
x = np.arange(len(donors))

for ax, result, title in [
    (axes[0], result_dirichlet, "Dirichlet (WeightedSumFitter)"),
    (axes[1], result_softmax, "Softmax-Normal (SoftmaxWeightedSumFitter)"),
]:
    median, lo, hi = _extract_weights(result, target_country)
    ax.barh(x, median, xerr=[median - lo, hi - median], height=0.6, capsize=3)
    ax.set_yticks(x)
    ax.set_yticklabels(donors)
    ax.set_xlabel("Weight")
    ax.set_title(title)
    ax.axvline(1 / len(donors), color="grey", ls="--", lw=0.8, label="1/N (uniform)")
    ax.legend(fontsize=8)

fig.suptitle("Posterior donor weights (median ± 94% HDI)", y=1.02)
fig.tight_layout()

### Counterfactual trajectories

Despite different weight distributions, both models should recover similar counterfactual trajectories for the UK — the synthetic control is a weighted sum, and multiple weight configurations can yield comparable fits.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Observed
ax.plot(df.index, df[target_country], "k-", lw=2, label="UK (observed)")

# Dirichlet counterfactual
pred_dir = result_dirichlet.post_pred.mean(dim=["chain", "draw"]).sel(
    treated_units=target_country
)
ax.plot(pred_dir.coords["obs_ind"], pred_dir.values, "C0--", lw=1.5, label="Dirichlet")

# Softmax-Normal counterfactual
pred_sm = result_softmax.post_pred.mean(dim=["chain", "draw"]).sel(
    treated_units=target_country
)
ax.plot(
    pred_sm.coords["obs_ind"], pred_sm.values, "C1--", lw=1.5, label="Softmax-Normal"
)

ax.axvline(treatment_time, color="r", ls=":", label="Brexit vote")
ax.set(ylabel="Trillion USD", title="Counterfactual comparison")
ax.legend()

In [ ]:
stats_dir = result_dirichlet.effect_summary(treated_unit=target_country)
stats_sm = result_softmax.effect_summary(treated_unit=target_country)

comparison = pd.concat(
    [
        stats_dir.table.assign(model="Dirichlet"),
        stats_sm.table.assign(model="Softmax-Normal"),
    ]
).set_index("model", append=True)
comparison

Both models recover similar counterfactuals and causal effect estimates, but the weight distributions differ. The softmax-Normal model lets us *tune* this behaviour via the prior scale `sigma` — the topic of the next section.

## The sigma knob: regularisation path

The prior scale `sigma` on the softmax-Normal logits controls where the model sits on the DiD-to-SC spectrum:

- **Small sigma** (e.g. 0.01): logits are shrunk toward zero → softmax produces near-uniform weights → DiD-like behaviour.
- **Large sigma** (e.g. 10): the prior is nearly flat → data drives weights toward a sparse solution → SC-like behaviour.
- **Default sigma = 1.0**: moderate regularisation, a sensible middle ground.

This is the Bayesian analogue of $1/\zeta$ in the frequentist SDiD of {cite:t}`arkhangelsky2021synthetic`. To show this, we re-fit `SoftmaxWeightedSumFitter` at six values of `sigma` and plot how the posterior weights change — a **regularisation path**, analogous to the coefficient path plots from ridge/LASSO regression.

In [ ]:
sigma_values = [0.01, 0.1, 0.5, 1.0, 3.0, 10.0]
path_results = {}

for sigma in sigma_values:
    print(f"Fitting sigma={sigma}...")
    model = cp.pymc_models.SoftmaxWeightedSumFitter(
        sample_kwargs=sample_kwargs,
        priors={
            "beta_raw": Prior(
                "Normal",
                mu=0,
                sigma=sigma,
                dims=["treated_units", "coeffs_raw"],
            )
        },
    )
    path_results[sigma] = cp.SyntheticControl(
        df,
        treatment_time,
        control_units=other_countries,
        treated_units=[target_country],
        model=model,
    )

In [ ]:
# Collect posterior weight statistics across sigma values
medians = np.zeros((len(sigma_values), len(other_countries)))
lows = np.zeros_like(medians)
highs = np.zeros_like(medians)

for i, sigma in enumerate(sigma_values):
    beta = path_results[sigma].idata.posterior["beta"].sel(treated_units=target_country)
    medians[i] = beta.median(dim=["chain", "draw"]).values
    hdi = az.hdi(beta, hdi_prob=0.94)["beta"].values
    lows[i] = hdi[:, 0]
    highs[i] = hdi[:, 1]

# Identify top donors (by max median weight across sigma values)
max_weights = medians.max(axis=0)
top_k = 5
top_indices = np.argsort(max_weights)[-top_k:]
top_donors = [other_countries[j] for j in top_indices]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

for j, donor in enumerate(other_countries):
    is_top = j in top_indices
    color = f"C{list(top_indices).index(j)}" if is_top else "grey"
    alpha = 1.0 if is_top else 0.3
    lw = 1.5 if is_top else 0.8
    label = donor if is_top else (None if j > 0 else "Other donors")

    ax.plot(sigma_values, medians[:, j], color=color, alpha=alpha, lw=lw, label=label)

    if is_top:
        ax.fill_between(sigma_values, lows[:, j], highs[:, j], color=color, alpha=0.15)

# Reference line at default sigma
ax.axvline(1.0, color="black", ls="--", lw=0.8, alpha=0.5, label="Default (σ=1)")

# Uniform reference
ax.axhline(1 / len(other_countries), color="grey", ls=":", lw=0.8, alpha=0.5)

ax.set_xscale("log")
ax.set_xlabel("σ (prior scale on logits)")
ax.set_ylabel("Posterior median weight")
ax.set_title("Regularisation path: softmax-Normal weights across σ")

# Annotations
ax.annotate(
    "← DiD-like\n(uniform)",
    xy=(0.012, 1 / len(other_countries)),
    fontsize=9,
    ha="left",
    va="bottom",
    color="grey",
)
ax.annotate(
    "SC-like →\n(sparse)",
    xy=(8, medians[-1].max()),
    fontsize=9,
    ha="right",
    va="bottom",
    color="grey",
)

ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()

At small `sigma` (left), all donors receive roughly equal weight (~1/15) — the prior dominates and the model behaves like DiD. As `sigma` increases, the data gets more influence: a few donors accumulate weight while others shrink toward zero, mimicking a sparse SC solution. The credible ribbons confirm that the posterior is tight under strong regularisation and wide when the data drives the weights.

The vertical dashed line at `sigma=1.0` shows that the default sits in a sensible middle ground — some differentiation between donors, but no extreme sparsity.